# Radom Forest Classifier (RF)

In [1]:
import time
import sys
import pandas as pd
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Random Forrest (RF) ----

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n10000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
rfc = RandomForestClassifier()

start_time = time.process_time()
rfc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = rfc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = rfc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(rfc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

Accuracy         0.9245
Precision        0.9293
Recall           0.9248
F1-Score         0.9255
Training Time    7.7031 s
Size in KB       17588.5459 KB


In [16]:
# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
rfc = RandomForestClassifier(random_state=0,
                             n_estimators=500,
                             max_samples=0.1,
                             criterion="log_loss",
                             min_samples_leaf=10,
                             max_depth=15,
                             n_jobs=-1,
                             ccp_alpha=0.001
                             )

start_time = time.process_time()
rfc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = rfc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = rfc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(rfc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

Accuracy         0.9193
Precision        0.9329
Recall           0.9193
F1-Score         0.9222
Training Time    1220.9062 s
Size in KB       22087.0654 KB


In [32]:
# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
rfc = RandomForestClassifier(random_state=0,
                             n_estimators=150,
                             max_samples=0.15,
                             criterion="entropy",
                             min_samples_leaf=50,
                             max_depth=20,
                             n_jobs=-1
                             )

start_time = time.process_time()
rfc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = rfc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = rfc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(rfc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

Accuracy         0.9424
Precision        0.9487
Recall           0.9424
F1-Score         0.9438
Training Time    922.2969 s
Size in KB       26626.0732 KB


120, 0.10
Accuracy         0.9363
Precision        0.9438
Recall           0.9364
F1-Score         0.938
Training Time    492.0156 s
Size in KB       15938.4443 KB

***
## Model Optimization

***
### Random Search with Cross Validation

In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.model_selection import RandomizedSearchCV

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

rfc = RandomForestClassifier(
    random_state=0,
    n_jobs=-1
)

param_distribution = {
    'n_estimators': [100, 200, 300, 500],
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 20, 30, None],                    # Control RAM/Overfitting
    'min_samples_leaf': [1, 2, 4, 10, 50],              # Higher values = smaller, faster trees
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],    # How many features to look at per split
    'max_leaf_nodes': [None],
    'max_samples': [0.1, 0.15, 0.2, 0.3, 0.5],
    'bootstrap': [True, False],                         # Whether to use subsamples
    'warm_start': [True, False]                         
}

rfc_random = RandomizedSearchCV(
    estimator=rfc,
    param_distributions=param_distribution,
    n_iter=50,           # Number of random combinations to try
    cv=3,                # 3-fold cross validation
    verbose=2, 
    random_state=0,
    scoring='accuracy',
    refit=True,
    n_jobs=-1            # Use all cores for the search itself
)

start_wall_time = time.perf_counter()
rfc_random.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_rfc = rfc_random.best_estimator_

y_pred = best_rfc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_rfc)) / 1024
training_time = rfc_random.cv_results_['mean_fit_time'][rfc_random.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {rfc_random.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {rfc_random.best_score_:.4f}")

Fitting 3 folds for each of 50 candidates, totalling 150 fits


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Accuracy         0.9674
Precision        0.9693
Recall           0.9673
F1-Score         0.9677
Training Time    702.479 s
Size in KB       50302.4893 KB

[+] Best Parameters: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': None, 'criterion': 'entropy', 'bootstrap': True}
[+] Best Cross-Validation Accuracy: 0.9635

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Accuracy         0.9876
Precision        0.988
Recall           0.9876
F1-Score         0.9877
Training Time    1953.3702 s
Size in KB       147470.2529 KB

[+] Best Parameters: {
    'warm_start': True,
    'n_estimators': 100,
    'min_samples_leaf': 2,
    'max_samples': 0.5,
    'max_leaf_nodes': None,
    'max_features': 0.5, 
    'max_depth': None, 
    'criterion': 'entropy', 
    'bootstrap': True}
[+] Best Cross-Validation Accuracy: 0.9635

***
## Final Model Test

In [13]:
import time
import sys
import pandas as pd
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Random Forrest (RF) ----

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000_shuffled.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
rfc = RandomForestClassifier(
    random_state=0,
    warm_start=True,
    n_estimators=100,
    min_samples_leaf=2,
    max_samples=0.1,
    max_leaf_nodes=None,
    max_features=0.2,     # More features per tree exponentially increases training time
    max_depth= None,
    criterion='entropy',
    bootstrap=True,
    n_jobs=-1
)

start_time = time.process_time()
rfc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = rfc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = rfc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(rfc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

Accuracy         0.9732
Precision        0.9749
Recall           0.9733
F1-Score         0.9736
Training Time    523.6094 s
Size in KB       75394.252 KB


In [14]:
estimators = rfc.estimators_

for e in estimators:
    print(e.get_depth())

for e in estimators:
    print(e.get_n_leaves())

31
34
31
30
31
36
35
32
34
32
31
32
32
34
35
38
34
33
36
31
33
31
36
41
29
29
35
37
32
33
30
32
37
32
38
32
33
32
30
36
32
36
32
33
30
36
31
31
31
33
34
33
32
32
33
34
31
36
32
32
34
31
31
35
32
30
36
32
33
34
33
35
34
32
32
33
33
28
35
33
33
30
31
35
32
35
36
33
31
35
30
32
32
33
33
35
35
42
31
35
2751
3035
3001
2943
3039
3101
3148
3106
3085
3016
3375
2992
2946
3098
3304
3021
3081
2827
2995
2916
3040
2951
2966
3071
2976
3000
2863
2807
3044
2953
3119
3055
3008
2860
3061
2839
3063
2979
3081
2931
3048
2891
2982
2779
2933
3014
3039
3015
2983
3263
3023
3039
3036
2779
3113
3056
3076
2954
3212
2785
2975
3421
3098
3041
3096
3001
3085
3090
3072
2924
2842
3125
3076
3099
2875
3075
2986
2860
3184
2955
2928
2969
3413
3139
2840
2888
2983
3128
3044
2870
2841
2918
3342
2914
2888
3055
3059
2889
3185
2823


In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Random Forrest (RF) ----

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000_shuffled.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
rfc = RandomForestClassifier(
    random_state=0,
    warm_start=True,
    n_estimators=100,
    min_samples_leaf=2,
    max_samples=0.5,
    max_leaf_nodes=None,
    max_features=0.5,     # More features per tree exponentially increases training time
    max_depth=None,
    criterion='entropy',
    bootstrap=True,
    n_jobs=-1
)

start_time = time.process_time()
rfc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
start_time = time.process_time()
y_pred = rfc.predict(X_test)
predict_time = time.process_time() - start_time

report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = rfc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(rfc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Predict Time     {round(predict_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

with open("rfc_load_tester.pkl", 'wb') as file:
    pickle.dump(rfc, file)
print(f"Model successfully saved to rfc_load_tester.pkl")

Accuracy         0.9871
Precision        0.9874
Recall           0.9871
F1-Score         0.9871
Training Time    4987.7656 s
Size in KB       148517.752 KB
Model successfully saved to rfc_load_tester.pkl


Accuracy         0.9761
Precision        0.9774
Recall           0.9762
F1-Score         0.9764
Training Time    1750.7656 s
Size in KB       56383.252 KB

**Base Model:**
Accuracy         0.9245
Precision        0.9293
Recall           0.9248
F1-Score         0.9255
Training Time    7.7031 s
Size in KB       17588.5459 KB

**Final Model:**
Accuracy         0.9732
Precision        0.9749
Recall           0.9733
F1-Score         0.9736
Training Time    494.25 s
Size in KB       75394.252 KB

Final model has a 4.87% better, while being 4.2 times bigger and taking 64 times longer to train.

